<a href="https://colab.research.google.com/github/Muneebshah1192/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## My Rule

I will prioritize content for refreshing based on three observable signals:

1. Pages with higher search volume have greater opportunity to gain additional traffic.
2. Pages that have not been updated for a long time are more likely to benefit from a content refresh.
3. Pages with lower CTR may have opportunities to improve titles, meta descriptions, or overall relevance.

The baseline score combines these three signals into a single ranking score. Pages with higher scores are recommended for content refresh.

### Reason Code

**STALE_HIGH_VOLUME_LOW_CTR**

Meaning:
- STALE → The content has not been updated recently.
- HIGH_VOLUME → The target keyword has meaningful search demand.
- LOW_CTR → The page has relatively low click-through rate, indicating possible optimization opportunities.

### Action Label

**REFRESH_CONTENT**

Pages with higher baseline scores are assigned the action **REFRESH_CONTENT** because they represent the best candidates for improving search performance.

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [6]:
!git clone https://github.com/Muneebshah1192/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 130 (delta 39), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 1.87 MiB | 8.42 MiB/s, done.
Resolving deltas: 100% (39/39), done.


In [7]:
import os
os.listdir("/content")

['.config', 'flyrank-ml-internship', 'sample_data']

In [8]:
import os

os.listdir("/content/flyrank-ml-internship/data/raw")

['content_refresh_anonymized.csv']

In [9]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)

df.head()

Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,NaN,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,NaN,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,91-180,4,20,0-30,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,NaN,NaN,11751,58,87,78,75,1,0,3,88,51,3626,22,35,4206,17,26,463,365+,6,22,0-30,NaN,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,NaN,gemini-3-flash-preview,19140,24,177,145,144,0,0,43,88,33,4211,10,14,6452,2,9,263,181-365,5,14,0-30,2000-3500,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [10]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

In [12]:
df.describe()

,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier_order,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,trend_pct
count,27532.000000,27532.000000,27532.000000,22301.000000,22301.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.00000,30000.000000,30000.000000,30000.000000,30000.00000,30000.000000,29875.000000,30000.000000,26612.000000
mean,158.882391,0.146954,0.485342,3107.760325,20665.277835,5200.366300,16.097333,49.942467,37.066633,35.937700,0.991933,0.204500,4.033500,61.454033,13.096333,1429.058733,4.933867,14.114267,1783.078500,5.435100,10.283000,256.16780,4.786533,46.098300,0.510733,16.34238,2.534520,18.212921,0.768196,-4.785969
std,1518.270825,0.285241,2.101560,1452.382598,10115.344042,16838.019547,75.076958,152.101430,107.069131,103.748185,4.359576,1.363601,22.335179,32.689245,17.307759,5643.852081,23.929393,39.036371,6150.429511,28.358673,42.578003,132.70793,0.790392,42.078709,3.279162,15.21679,8.310096,29.472768,7.429454,473.861780
min,0.000000,0.000000,0.000000,8.000000,40.000000,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,90.00000,3.000000,1.000000,0.000000,0.00000,0.000000,0.000000,0.000000,-100.000000
25%,0.000000,0.000000,0.000000,2413.000000,15644.000000,81.000000,0.000000,2.000000,2.000000,2.000000,0.000000,0.000000,0.000000,31.000000,2.000000,10.000000,0.000000,0.000000,19.000000,0.000000,1.000000,132.00000,4.000000,20.000000,0.000000,6.20000,0.000000,0.000000,0.000000,-62.600000
50%,10.000000,0.000000,0.000000,2877.000000,19116.000000,731.000000,1.000000,8.000000,7.000000,7.000000,0.000000,0.000000,1.000000,81.000000,6.000000,139.000000,0.000000,3.000000,210.000000,0.000000,2.000000,236.00000,5.000000,20.000000,0.070000,10.80000,0.000000,5.000000,0.000000,-33.500000
75%,20.000000,0.130000,0.000000,3666.000000,24011.000000,3615.250000,7.000000,33.000000,27.000000,27.000000,1.000000,0.000000,3.000000,88.000000,16.000000,768.000000,2.000000,12.000000,1143.000000,2.000000,7.000000,333.00000,5.000000,104.000000,0.290000,22.30000,1.350000,23.530000,0.000000,0.000000
max,74000.000000,1.000000,100.360000,9546.000000,111158.000000,517715.000000,4178.000000,5998.000000,4345.000000,4913.000000,290.000000,64.000000,2605.000000,88.000000,90.000000,238796.000000,1176.000000,1081.000000,218786.000000,1627.000000,4247.000000,564.00000,6.000000,373.000000,100.000000,245.00000,100.000000,300.000000,300.000000,44900.000000


In [13]:
missing = df.isnull().sum().sort_values(ascending=False)

missing[missing > 0]

,0
provider_used,21438
word_count,7699
char_count,7699
word_count_tier,7699
char_count_tier,7699
model_used,5733
trend_pct,3388
competition_level,2610
search_volume,2468
cpc,2468


In [14]:
signal1 = (
    df.groupby("freshness_tier")
      .agg(
          n=("content_id", "count"),
          avg_ctr=("ctr", "mean"),
          avg_impressions=("impressions_90d", "mean"),
          avg_sessions=("sessions_90d", "mean")
      )
)

signal1

,n,avg_ctr,avg_impressions,avg_sessions
freshness_tier,,,,
0-30,20480,0.609021,4199.614062,24.453662
181+,174,3.693276,1172.448276,5.540230
31-90,175,0.117543,6506.748571,28.171429
91-180,9171,0.238367,7486.665140,66.000872


### Signal Check 1 – Freshness

Signal:
Days Since Last Update (FlyRank Refresh Flag)

Bucket Table:
(Generated above)

Verdict:
**MIXED**

Reason:
The relationship between freshness and performance is not consistent.
Pages with very old content (181+ days) have the highest average CTR but the
lowest impressions and sessions. Meanwhile, pages updated 91–180 days ago
receive the highest impressions. Therefore, freshness is useful but should not
be the only signal in the baseline rule.

In [15]:
bins = [0, 100, 1000, 5000, 10000, 100000]
labels = [
    "0-100",
    "101-1k",
    "1k-5k",
    "5k-10k",
    "10k+"
]

df["volume_bucket"] = pd.cut(
    df["search_volume"],
    bins=bins,
    labels=labels
)

signal2 = (
    df.groupby("volume_bucket")
      .agg(
          n=("content_id", "count"),
          avg_impressions=("impressions_90d", "mean"),
          avg_clicks=("clicks_90d", "mean"),
          avg_ctr=("ctr", "mean")
      )
)

signal2

/tmp/ipykernel_3629/1588272054.py:17: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("volume_bucket")


,n,avg_impressions,avg_clicks,avg_ctr
volume_bucket,,,,
0-100,13402,5359.707059,17.337487,0.255963
101-1k,2489,5512.677782,13.560064,0.182081
1k-5k,408,6894.909314,12.237745,0.167868
5k-10k,85,3153.047059,2.541176,0.080235
10k+,67,8231.656716,8.701493,0.397910


### Signal Check 2 – Search Volume

Signal:
Search Volume

Bucket Table:
(Generated above)

Verdict:
**CONFIRMED**

Reason:
Pages targeting keywords with larger search volume generally receive more
impressions than pages targeting low-volume keywords. Although one bucket
(5k–10k) performs below the surrounding groups, the overall trend supports
search volume as an opportunity signal.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Baseline Action Score

The baseline score combines three safe signals:

- Search Volume (higher = more opportunity)
- Days Since Last Update (older = more likely to need refresh)
- CTR (lower CTR = greater optimization opportunity)

The score is used only for ranking pages and is not a machine learning model.

Reason Code:
STALE_HIGH_VOLUME_LOW_CTR

Action:
REFRESH_CONTENT

In [16]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# Fill missing search volume
df["search_volume_fill"] = df["search_volume"].fillna(0)

# Normalize features
df["sv_norm"] = scaler.fit_transform(df[["search_volume_fill"]])

df["freshness_norm"] = scaler.fit_transform(df[["days_since_last_update"]])

df["ctr_inverse"] = 1 - scaler.fit_transform(df[["ctr"]])

# Baseline score
df["baseline_score"] = (
    0.4 * df["sv_norm"] +
    0.4 * df["freshness_norm"] +
    0.2 * df["ctr_inverse"]
)

# Reason code
df["reason_code"] = "STALE_HIGH_VOLUME_LOW_CTR"

# Action label
df["action"] = "REFRESH_CONTENT"

# Sort
ranked_queue = df.sort_values(
    by="baseline_score",
    ascending=False
)

ranked_queue.head(20)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,volume_bucket,search_volume_fill,sv_norm,freshness_norm,ctr_inverse,baseline_score,reason_code,action
12140,content_ef99c4abd9ab,client_3fdba35f04,74000.0,0.08,LOW,0.34,keyword article,informational,NaN,NaN,NaN,gpt-4o-mini,3016,1,20,11,10,0,0,1,88,6,812,1,4,784,0,1,463,365+,6,104,91-180,NaN,NaN,0.03,38.5,0.00,5.00,0.0,good,page_3_5,stable,3.6,10k+,74000.0,1.000000,0.276882,0.9997,0.710693,STALE_HIGH_VOLUME_LOW_CTR,REFRESH_CONTENT
28282,content_454cc6654c6e,client_3fdba35f04,60500.0,0.11,LOW,0.39,keyword article,informational,NaN,NaN,NaN,gpt-4o-mini,4364,0,21,19,19,0,0,1,88,13,650,0,10,1376,0,5,463,365+,6,104,91-180,NaN,NaN,0.00,44.9,0.00,4.76,0.0,good,page_3_5,down,-52.8,10k+,60500.0,0.817568,0.276882,1.0000,0.637780,STALE_HIGH_VOLUME_LOW_CTR,REFRESH_CONTENT
18701,content_deb54e9e19cd,client_3fdba35f04,60500.0,0.13,LOW,0.56,keyword article,informational,NaN,NaN,NaN,gpt-4o-mini,4560,0,9,6,6,0,0,0,88,3,1119,0,6,1148,0,0,463,365+,6,104,91-180,NaN,NaN,0.00,41.7,0.00,0.00,0.0,good,page_3_5,stable,-2.5,10k+,60500.0,0.817568,0.276882,1.0000,0.637780,STALE_HIGH_VOLUME_LOW_CTR,REFRESH_CONTENT
17907,content_5ec29ae79c60,client_3fdba35f04,60500.0,0.13,LOW,0.76,keyword article,informational,NaN,NaN,NaN,gpt-4o-mini,2439,0,6,6,6,0,0,0,88,5,905,0,5,523,0,1,463,365+,6,104,91-180,NaN,NaN,0.00,49.8,0.00,0.00,0.0,moderate,page_3_5,up,73.0,10k+,60500.0,0.817568,0.276882,1.0000,0.637780,STALE_HIGH_VOLUME_LOW_CTR,REFRESH_CONTENT
6972,content_bf67a444faef,client_3fdba35f04,60500.0,0.11,LOW,0.50,keyword article,informational,NaN,NaN,NaN,gpt-4o-mini,3141,0,15,13,13,0,0,2,88,7,531,0,1,664,0,2,463,365+,6,104,91-180,NaN,NaN,0.00,45.5,0.00,13.33,0.0,good,page_3_5,down,-20.0,10k+,60500.0,0.817568,0.276882,1.0000,0.637780,STALE_HIGH_VOLUME_LOW_CTR,REFRESH_CONTENT
29384,content_f6fdf87348f6,client_4ec9599fc2,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,NaN,gpt-5-mini,2,0,1,1,1,0,0,0,1,1,0,0,1,2,0,0,373,365+,6,373,181+,NaN,NaN,0.00,32.5,0.00,0.00,0.0,low,page_3_5,down,-100.0,NaN,0.0,0.000000,1.000000,1.0000,0.600000,STALE_HIGH_VOLUME_LOW_CTR,REFRESH_CONTENT
26242,content_55a5b1c46474,client_4ec9599fc2,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,NaN,gpt-5-mini,35,0,1,1,1,0,0,0,21,1,3,0,0,26,0,1,374,365+,6,373,181+,NaN,NaN,0.00,7.5,0.00,0.00,0.0,low,page_1,down,-88.5,NaN,0.0,0.000000,1.000000,1.0000,0.600000,STALE_HIGH_VOLUME_LOW_CTR,REFRESH_CONTENT
24216,content_1b4ec72dafd4,client_4ec9599fc2,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,NaN,gpt-5-mini,2,0,2,2,2,0,0,1,1,2,0,0,0,2,0,1,372,365+,6,372,181+,NaN,NaN,0.00,7.0,0.00,50.00,0.0,low,page_1,down,-100.0,NaN,0.0,0.000000,0.997312,1.0000,0.598925,STALE_HIGH_VOLUME_LOW_CTR,REFRESH_CONTENT
18440,content_8d56efff1e71,client_4ec9599fc2,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,NaN,gpt-5-mini,1,0,2,2,2,0,0,1,1,2,1,0,0,0,0,1,372,365+,6,372,181+,NaN,NaN,0.00,35.0,0.00,50.00,0.0,low,page_3_5,new,NaN,NaN,0.0,0.000000,0.997312,1.0000,0.598925,STALE_HIGH_VOLUME_LOW_CTR,REFRESH_CONTENT
8631,content_e2b702f4f92b,client_4ec9599fc2,NaN,NaN,NaN,NaN,keyword article,NaN,1246.0,8740.0,NaN,gpt-5-mini,30,0,4,4,4,0,0,2,19,3,6,0,0,22,0,1,334,181-365,5,334,181+,1000-2000,8000-15000,0.00,9.3,0.00,50.00,0.0,low,page_1,down,-72.7,NaN,0.0,0.000000,0.895161,1.0000,0.558065,STALE_HIGH_VOLUME_LOW_CTR,REFRESH_CONTENT


In [17]:
import os

os.makedirs("/content/flyrank-ml-internship/work/outputs", exist_ok=True)

ranked_queue.to_csv(
    "/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully!")

CSV saved successfully!


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [18]:
top20 = ranked_queue.head(20)

top20[[
    "content_id",
    "baseline_score",
    "action",
    "reason_code",
    "days_since_last_update",
    "search_volume",
    "ctr"
]]

,content_id,baseline_score,action,reason_code,days_since_last_update,search_volume,ctr
12140,content_ef99c4abd9ab,0.710693,REFRESH_CONTENT,STALE_HIGH_VOLUME_LOW_CTR,104,74000.0,0.03
28282,content_454cc6654c6e,0.637780,REFRESH_CONTENT,STALE_HIGH_VOLUME_LOW_CTR,104,60500.0,0.00
18701,content_deb54e9e19cd,0.637780,REFRESH_CONTENT,STALE_HIGH_VOLUME_LOW_CTR,104,60500.0,0.00
17907,content_5ec29ae79c60,0.637780,REFRESH_CONTENT,STALE_HIGH_VOLUME_LOW_CTR,104,60500.0,0.00
6972,content_bf67a444faef,0.637780,REFRESH_CONTENT,STALE_HIGH_VOLUME_LOW_CTR,104,60500.0,0.00
29384,content_f6fdf87348f6,0.600000,REFRESH_CONTENT,STALE_HIGH_VOLUME_LOW_CTR,373,0.0,0.00
26242,content_55a5b1c46474,0.600000,REFRESH_CONTENT,STALE_HIGH_VOLUME_LOW_CTR,373,0.0,0.00
24216,content_1b4ec72dafd4,0.598925,REFRESH_CONTENT,STALE_HIGH_VOLUME_LOW_CTR,372,0.0,0.00
18440,content_8d56efff1e71,0.598925,REFRESH_CONTENT,STALE_HIGH_VOLUME_LOW_CTR,372,0.0,0.00
8631,content_e2b702f4f92b,0.558065,REFRESH_CONTENT,STALE_HIGH_VOLUME_LOW_CTR,334,NaN,0.00


In [19]:
for i, row in top20.iterrows():

    print(f"""
Content ID: {row['content_id']}

Action:
REFRESH_CONTENT

Reason Code:
STALE_HIGH_VOLUME_LOW_CTR

Confidence:
Medium

Why selected:
The page has a relatively high baseline score because it combines high search opportunity,
older content, and lower CTR.

What would make it wrong?
The page may already have been recently improved, or low CTR may be caused by factors
outside the page content such as SERP competition.

------------------------------------------------------------
""")


Content ID: content_ef99c4abd9ab

Action:
REFRESH_CONTENT

Reason Code:
STALE_HIGH_VOLUME_LOW_CTR

Confidence:
Medium

Why selected:
The page has a relatively high baseline score because it combines high search opportunity,
older content, and lower CTR.

What would make it wrong?
The page may already have been recently improved, or low CTR may be caused by factors
outside the page content such as SERP competition.

------------------------------------------------------------


Content ID: content_454cc6654c6e

Action:
REFRESH_CONTENT

Reason Code:
STALE_HIGH_VOLUME_LOW_CTR

Confidence:
Medium

Why selected:
The page has a relatively high baseline score because it combines high search opportunity,
older content, and lower CTR.

What would make it wrong?
The page may already have been recently improved, or low CTR may be caused by factors
outside the page content such as SERP competition.

------------------------------------------------------------


Content ID: content_deb54e9e19cd

A

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some highly ranked pages may not actually require refreshing.

Examples include:

- Pages with extremely high search volume but already performing well.
- Pages with low CTR because of strong SERP competition rather than poor content.
- Pages with limited traffic despite being old.

These should be manually reviewed before taking action.

## Leakage Check

No identifier columns were used.

Not used:

- trend_direction
- trend_pct
- content_id
- client_id
- provider_used
- model_used

The baseline only uses historical features available at scoring time.

No future-window information or label-derived features were included.

In [21]:
import os

os.makedirs("/content/flyrank-ml-internship/work/outputs", exist_ok=True)

In [22]:
df.to_csv(
    "/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv",
    index=False
)

In [23]:
import os

os.listdir("/content/flyrank-ml-internship/work/outputs")

['baseline_action_score.csv']

In [24]:
from google.colab import files

files.download("/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
!git status

fatal: not a git repository (or any of the parent directories): .git


In [26]:
%cd /content/flyrank-ml-internship

/content/flyrank-ml-internship


In [27]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [28]:
import os

os.listdir("/content/flyrank-ml-internship/work/outputs")

['baseline_action_score.csv']

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.